## 2.3 PIIMiddleware中间件
敏感信息保护。

PII中间件用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。

```python
PIIMiddleware(
  self,
  pii_type: Literal['email', 'credit_card', 'ip', 'mac_address', 'url'] | str,
  *,
  strategy: Literal['block', 'redact', 'mask', 'hash'] = 'redact',
  detector: Callable[[str], list[PIIMatch]] | str | None = None,
  apply_to_input: bool = True,
  apply_to_output: bool = False,
  apply_to_tool_results: bool = False
)
```

### 2.3.1 参数说明

参数1：pii_type — 检测的PII数据类型
可以是内置类型或自定义类型，自定义类型有
* email ：电子邮箱地址
* credit_card ：信用卡号
* url ：网址
* mac_address ：设备MAC地址
* ip ：IP地址

参数2：strategy —处理PII信息的策略

支持四种选项：
1. redact ：将检测到的PII信息用字符串 [REDACTED_[PII_TYPE]] 替换，其中的 PII_TYPE是上面提到的具体类型，比如 [REDACTED_EMAIL] 、 [REDACTED_CREDIT_CARD] 这样的标签。完全 “擦除/隐藏” 真实内容。适合日志清洗、合规需求、公开输出时隐藏敏感内容。
2. mask ：用 *** 将PII信息的前面一部分信息遮蔽。比如信用卡号可能变成 ****-****-****-1234 （只保留最后几位/部分可见），邮箱可能保留域名部分 + 隐藏用户名的一部分等 — 既隐藏大部分敏感信息，又保留了一点“可辨识性”（比如账号后四位、域名等），适合用户服务界面/ 前端显示 / 需要部分可识别但不泄露完整敏感内容的场景。
3. hash ：用检测到的PII信息的 哈希值 替代原值。比如 <email_hash:a1b2c3d4> 。适合analytics、调试 (debug)、统计分析、匿名追踪等场景。
4. block ：如果检测到PII信息， 直接抛出异常 。适合对隐私要求极高、绝不允许泄露任何敏感信息的场景。

参数3：detector —自定义 PII检测函数 或者 正则表达式

如果没有提供则使用内置的检测函数。

LangChain为每种PII信息定制了专门的检测函数，相关源码如下

```python
BUILTIN_DETECTORS: dict[str, Detector] = {
    "email": detect_email,
    "credit_card": detect_credit_card,
    "ip": detect_ip,
    "mac_address": detect_mac_address,
    "url": detect_url,
}
```

参数4：apply_to_input —是否在调用模型前检测

默认为True。

参数5：apply_to_output —是否在模型调用后检测

默认为False。

参数6：apply_to_tool_results —是否在工具调用后检测其输出

默认为False。



通常我们 只在模型调用前 检测。因为PII检测的主要目的是避免将敏感信息发送给模型服务导致信息泄露。

### 举例1：使用内置检测器

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [
        HumanMessage("""
        帮我向 156168188@qq.com 发送一封邮件
        同时查看银行卡号： 5105-1051-0510-5100 的余额
        访问 https://localhost:12345
        确认这是不是 MAC地址： 11-11-11-11-11-11
        """)
    ]
})

In [3]:
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


        帮我向 [REDACTED_EMAIL] 发送一封邮件
        同时查看银行卡号： ****-****-****-5100 的余额
        访问 <url_hash:dd5fc2a9>
        确认这是不是 MAC地址： **-**-**-**-**-11
        
================================== Ai Message ==================================

我目前无法直接发送邮件、登录银行查询余额，或访问这个被隐藏的链接。

- **邮件**：请提供邮件主题和正文（无需提供银行卡信息），我可以帮你拟好。
- **银行卡余额**：请通过银行官方 App、官网或客服电话查询；不要在聊天中发送完整卡号、密码或验证码。
- **链接**：我看到的是隐藏标记 `<url_hash:dd5fc2a9>`，不是可打开的网址。请确认它是否为安全、可信的链接，再通过浏览器访问。
- **MAC 地址**：`**-**-**-**-**-11` 被遮挡，无法确认是不是有效地址。标准格式是 6 组两位十六进制数，以连字符或冒号分隔；需要未遮挡的格式才能判断，但请勿分享与你无关的设备标识。


In [4]:
try:
    response1 = agent.invoke({"messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]})
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


### 2.3.3 举例2：自定义检测器/函数

自定义检测函数

In [5]:
import re
# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如"13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True,detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True,detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [
        HumanMessage("""
            这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
            帮我给这个号码打电话： 12345612345
            访问 https://localhost:12345
        """)
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


            这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
            帮我给这个号码打电话： ****2345
            访问 https://localhost:12345
        
================================== Ai Message ==================================

`<api_key_hash:6c678cc0>` 看起来是脱敏后的占位符或哈希标识，不是可直接使用的 API key；仅凭它无法确认密钥是否有效。请在服务端通过实际 API 请求验证，并避免公开真实密钥。

我不能替你拨打电话。`https://localhost:12345` 指向运行在你自己设备上的本地服务，我无法从这里访问；你可以在本机浏览器中打开它。
